# W5: SNS 콘텐츠 자동화

이 노트북은 F&B 매장을 위한 SNS 콘텐츠를 자동으로 생성하는 도구입니다.

## Step 0: 라이브러리 설치 및 Gemini 설정

In [ ]:
import subprocess
subprocess.run(['apt-get','-qq','-y','install','fonts-nanum'],capture_output=True)
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
_n=[f for f in fm.findSystemFonts() if 'Nanum' in f]
if _n: fm.fontManager.addfont(_n[0]); plt.rcParams['font.family']=fm.FontProperties(fname=_n[0]).get_name()
plt.rcParams['axes.unicode_minus']=False; print('✅ 폰트 설정 완료')


In [ ]:
# 필요한 라이브러리 설치
!pip install google-generativeai pandas -q

In [ ]:
import google.generativeai as genai
import pandas as pd
import json
import re
import zipfile
import os
from datetime import datetime
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

# Gemini API 설정
try:
    # 사용자에게 API 키 요청
    api_key = input("Google Gemini API 키를 입력하세요: ").strip()
    
    if api_key:
        genai.configure(api_key=api_key)
# === 2026년 3월 기준 최신 모델 선택 ===
# ✅ GA(안정): gemini-2.5-flash  ← 기본값 (권장)
# 🔵 Preview : gemini-3.1-flash-lite-preview  (최신, 2026.3 출시)
# 🔵 Preview : gemini-3.1-pro-preview  (최강, 복잡한 분석용)
# ⚠️  구버전 gemini-2.0-flash 는 2026년 6월 1일 종료 예정
MODEL_NAME = 'gemini-2.5-flash'  # 원하는 모델로 변경하세요
        model = genai.GenerativeModel('gemini-2.5-flash')
        print("✅ Gemini 2.0 Flash 모델 설정 완료")
        gemini_ready = True
    else:
        print("⚠️ API 키가 입력되지 않았습니다. 샘플 콘텐츠를 생성합니다.")
        gemini_ready = False
except Exception as e:
    print(f"❌ Gemini 설정 실패: {e}")
    print("샘플 콘텐츠를 생성합니다.")
    gemini_ready = False

## Step 1: 가게 정보 및 추천 메뉴 입력

In [ ]:
# 가게 정보 입력
store_info = {
    'name': input("가게 이름을 입력하세요: "),
    'type': input("음식점 종류를 입력하세요 (예: 한식, 양식, 중식, 일식): "),
    'specialty': input("특징이나 강점을 입력하세요 (예: 신선한 재료, 정통 레시피, 친절한 서비스): "),
    'location': input("위치를 입력하세요 (예: 강남역, 홍대): ")
}

# 이번 주 추천 메뉴 입력
print("\n이번 주 추천 메뉴를 입력하세요 (쉼표로 구분):")
recommended_menus = input("예: 김치찌개, 비빔밥, 불고기: ").split(',')
recommended_menus = [menu.strip() for menu in recommended_menus if menu.strip()]

# 가게 정보 표시
print(f"\n🏪 {store_info['name']} 정보")
for key, value in store_info.items():
    print(f"- {key}: {value}")
print(f"\n🌟 이번 주 추천 메뉴: {', '.join(recommended_menus)}")

## Step 2: 인스타그램 게시물 생성

In [ ]:
def generate_instagram_content(store_info, menus, use_gemini=True):
    """인스타그램 게시물 생성"""
    if use_gemini and gemini_ready:
        try:
            prompt = f"""
            다음 정보를 바탕으로 인스타그램 게시물 5개를 생성해주세요.
            
            가게 정보:
            - 이름: {store_info['name']}
            - 종류: {store_info['type']}
            - 특징: {store_info['specialty']}
            - 위치: {store_info['location']}
            - 추천 메뉴: {', '.join(menus)}
            
            각 게시물은 다음 형식을 따라주세요:
            1. 매력적인 해시태그 포함
            2. 관련 이모지 사용
            3. 1~5번 게시물 번호로 시작
            4. 짧고 흥미로운 내용
            5. 행동 촉구 (방문, 주문 등)
            
            JSON 형식으로 응답해주세요:
            {{
              "posts": [
                {{"number": 1, "content": "게시물 내용"}},
                {{"number": 2, "content": "게시물 내용"}},
                ...
              ]
            }}
            """
            
            response = model.generate_content(prompt)
            # JSON 파싱 시도
            try:
                json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
                if json_match:
                    result = json.loads(json_match.group())
                    return result['posts']
            except:
                pass
                
            # JSON 파싱 실패 시 텍스트로 처리
            posts = []
            lines = response.text.split('\n')
            for i, line in enumerate(lines[:5], 1):
                if line.strip():
                    posts.append({
                        "number": i,
                        "content": line.strip()
                    })
            return posts
            
        except Exception as e:
            print(f"Gemini API 오류: {e}")
            return generate_sample_instagram_content(store_info, menus)
    else:
        return generate_sample_instagram_content(store_info, menus)

def generate_sample_instagram_content(store_info, menus):
    """샘플 인스타그램 콘텐츠 생성"""
    return [
        {
            "number": 1,
            "content": f"🍳 {store_info['name']}의 {menus[0] if menus else '시그니처 메뉴'}가 드디어 완성! 정통 {store_info['type']}의 맛을 제대로 느껴보세요 🔥\n\n📍 {store_info['location']}에서 만나요!\n\n#{store_info['name']} #{store_info['type']} #{store_info['location']}맛집 #오늘의메뉴"
        },
        {
            "number": 2,
            "content": f"✨ {store_info['specialty']}로 만드는 특별한 맛! {store_info['name']}에서만 즐길 수 있는 진정한 {store_info['type']}의 감동 💯\n\n이번 주 특별 메뉴: {', '.join(menus[:2]) if len(menus) >= 2 else menus[0] if menus else '시그니처 메뉴'}\n\n#{store_info['name']}맛집 #{store_info['type']}추천 #맛스타그램"
        },
        {
            "number": 3,
            "content": f"🌟 오늘의 점심 메뉴 추천! {store_info['name']}의 {menus[1] if len(menus) > 1 else menus[0] if menus else '인기 메뉴'} 어떠세요?\n\n신선한 재료로 정성껏 준비했어요 🥰\n\n⏰ 영업시간: 11:00-21:00\n\n#{store_info['location']}맛집 #점심메뉴 #{store_info['name']}"
        },
        {
            "number": 4,
            "content": f"🎉 고객 후기 BEST! '{store_info['specialty']}'에 감동한 손님들 🥺\n\n"맛있어서 눈물 날 뻔했어요!"\n"정말 진정성 있는 맛이에요."\n\n여러분도 직접 경험해보세요! 💕\n\n#{store_info['name']}후기 #{store_info['type']}맛집 #감동맛집"
        },
        {
            "number": 5,
            "content": f"🚨 금주 마감 임박! {store_info['name']}에서 {', '.join(menus[:3]) if len(menus) >= 3 else ', '.join(menus)} 특별 할인 진행 중!\n\n🎯 20% 할인 쿠폰 증정 (선착순 30분)\n\n📍 {store_info['location']} 바로가기 지도 🗺️\n\n#{store_info['name']}할인 #{store_info['type']}추천 #마감임박"
        }
    ]

# 인스타그램 게시물 생성
instagram_posts = generate_instagram_content(store_info, recommended_menus, gemini_ready)

print("📱 인스타그램 게시물 (5개)")
for post in instagram_posts:
    print(f"\n--- 게시물 {post['number']} ---")
    print(post['content'])

## Step 3: 네이버 블로그 리뷰 생성

In [ ]:
def generate_blog_reviews(store_info, menus, use_gemini=True):
    """네이버 블로그 리뷰 생성"""
    if use_gemini and gemini_ready:
        try:
            prompt = f"""
            다음 정보를 바탕으로 네이버 블로그 방문기 형식의 리뷰 2개를 생성해주세요.
            
            가게 정보:
            - 이름: {store_info['name']}
            - 종류: {store_info['type']}
            - 특징: {store_info['specialty']}
            - 위치: {store_info['location']}
            - 추천 메뉴: {', '.join(menus)}
            
            각 리뷰는 다음 형식을 따라주세요:
            1. 방문기를 담은 스토리텔링
            2. 구체적인 메뉴 후기
            3. 분위기와 서비스 평가
            4. 재방문 의사 표현
            5. 블로그 스타일의 문체
            
            JSON 형식으로 응답해주세요:
            {{
              "reviews": [
                {{"title": "리뷰 제목1", "content": "리뷰 내용1"}},
                {{"title": "리뷰 제목2", "content": "리뷰 내용2"}}
              ]
            }}
            """
            
            response = model.generate_content(prompt)
            
            try:
                json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
                if json_match:
                    result = json.loads(json_match.group())
                    return result['reviews']
            except:
                pass
                
            # JSON 파싱 실패 시 텍스트로 처리
            return generate_sample_blog_reviews(store_info, menus)
            
        except Exception as e:
            print(f"Gemini API 오류: {e}")
            return generate_sample_blog_reviews(store_info, menus)
    else:
        return generate_sample_blog_reviews(store_info, menus)

def generate_sample_blog_reviews(store_info, menus):
    """샘플 블로그 리뷰 생성"""
    return [
        {
            "title": f"[{store_info['location']}맛집] {store_info['name']}에서 느끼는 {store_info['type']}의 진정성 🔥",
            "content": f"""안녕하세요! 오늘은 {store_info['location']}에 위치한 {store_info['name']}에 다녀왔어요!

📍 위치: {store_info['location']}에서 도보 5분 거리
⏰ 방문 시간: 평일 점심 시간 (12:30)

오늘은 친구와 함께 {store_info['name']}에 방문했는데, 정말 만족스러운 경험이었어요.

🍽️ 주문 메뉴:
- {menus[0] if menus else '시그니처 메뉴'}: 10점 만점에 10점!
- {menus[1] if len(menus) > 1 else '사이드 메뉴'}: 깔끔한 맛

특히 {menus[0] if menus else '시그니처 메뉴'}는 정말 최고였어요. {store_info['specialty']}라는 말이 이해가 가더라고요. 진정성 있는 {store_info['type']}의 맛을 느낄 수 있었어요.

💰 가격: 2인 기준 25,000원 (합리적!)
🏠 인테리어: 깔끔하고 아늑한 분위기
👩‍🍳 서비스: 사장님 정말 친절하세요!

재방문 의사 100% 입니다! 다음에 또 오겠습니다 👍

#주소: {store_info['location']} #전화번호: 예약 권장
#영업시간: 11:00-21:00

#맛스타그램 #{store_info['location']}맛집 #{store_info['name']} #{store_info['type']}맛집"""
        },
        {
            "title": f"[재방문 각] {store_info['name']} 이번 주 {', '.join(menus[:2]) if len(menus) >= 2 else menus[0] if menus else '추천 메뉴'} 꿀맛 ✨",
            "content": f"""두 번째 방문한 {store_info['name']} 후기예요!

첫 방문했을 때 만족해서 이번 주말에 또 왔는데, 이번에도 실망시키지 않았네요~

🌟 이번 주 추천 메뉴:
{chr(10).join([f'- {menu}' for menu in menus[:3]]) if len(menus) >= 3 else chr(10).join([f'- {menu}' for menu in menus]) if menus else '- 시그니처 메뉴'}

이번에 새로 시도한 {menus[1] if len(menus) > 1 else '추천 메뉴'}도 정말 맛있었어요. {store_info['specialty']}인게 먹을 때마다 느껴지더라고요.

📸 사진 첨부했는데 보기만 해도 침샘 자극하죠? 실제로 더 맛있어요!

✨ 강점 추천:
1. 맛: 10/10 (정통 {store_info['type']} 맛)
2. 가격: 9/10 (합리적이에요)
3. 분위기: 9/10 (가족, 친구랑 오기 좋아요)
4. 서비스: 10/10 (사장님 최고!)

{store_info['location']} 근처에 계신 분들 꼭 드셔보세요!
외식 고민될 때 {store_info['name']} 강력 추천합니다 💯

#맛집후기 #{store_info['location']}맛집탐방 #{store_info['name']}재방문 #{store_info['type']}추천"""
        }
    ]

# 블로그 리뷰 생성
blog_reviews = generate_blog_reviews(store_info, recommended_menus, gemini_ready)

print("📝 네이버 블로그 리뷰 (2개)")
for i, review in enumerate(blog_reviews, 1):
    print(f"\n--- 리뷰 {i}: {review['title']} ---")
    print(review['content'])

## Step 4: 카카오채널 공지 생성

In [ ]:
def generate_kakao_notice(store_info, menus, use_gemini=True):
    """카카오채널 공지 생성"""
    if use_gemini and gemini_ready:
        try:
            prompt = f"""
            다음 정보를 바탕으로 카카오채널 공지 1개를 생성해주세요.
            
            가게 정보:
            - 이름: {store_info['name']}
            - 종류: {store_info['type']}
            - 특징: {store_info['specialty']}
            - 위치: {store_info['location']}
            - 추천 메뉴: {', '.join(menus)}
            
            공지는 다음 형식을 따라주세요:
            1. 친근하고 부드러운 톤
            2. 이모지 사용
            3. 중요 정보 강조
            4. 행동 촉구
            5. 간결하고 명확한 내용
            
            JSON 형식으로 응답해주세요:
            {{
              "title": "공지 제목",
              "content": "공지 내용"
            }}
            """
            
            response = model.generate_content(prompt)
            
            try:
                json_match = re.search(r'\{.*\}', response.text, re.DOTALL)
                if json_match:
                    result = json.loads(json_match.group())
                    return result
            except:
                pass
                
            return generate_sample_kakao_notice(store_info, menus)
            
        except Exception as e:
            print(f"Gemini API 오류: {e}")
            return generate_sample_kakao_notice(store_info, menus)
    else:
        return generate_sample_kakao_notice(store_info, menus)

def generate_sample_kakao_notice(store_info, menus):
    """샘플 카카오채널 공지 생성"""
    return {
        "title": f"🍽️ {store_info['name']} 이번 주 특별 소식!",
        "content": f"""안녕하세요! {store_info['name']}입니다 🌟

🎉 이번 주 추천 메뉴 알려드려요!
{chr(10).join([f'• {menu}' for menu in menus[:3]]) if len(menus) >= 3 else chr(10).join([f'• {menu}' for menu in menus]) if menus else '• 시그니처 메뉴'}

✨ 특별 이벤트 안내
이번 주 {menus[0] if menus else '시그니처 메뉴'} 주문 시
음료 1杯 서비스! 🥤

📍 찾아오시는 길
- 주소: {store_info['location']} 근처
- 영업시간: 11:00-21:00
- 전화: 예약 가능

📞 문의 및 예약
카카오톡으로 바로 문의주세요!
빠른 시간 내에 답변드릴게요 😊

감사합니다!
{store_info['name']} 드림 ❤️"""
    }

# 카카오채널 공지 생성
kakao_notice = generate_kakao_notice(store_info, recommended_menus, gemini_ready)

print("💬 카카오채널 공지 (1개)")
print(f"\n제목: {kakao_notice['title']}")
print(f"내용:\n{kakao_notice['content']}")

## Step 5: 전체 콘텐츠 ZIP 다운로드

In [ ]:
# 파일 이름 생성
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_filename = f"{store_info['name']}_SNS콘텐츠_{timestamp}.zip"

# 텍스트 파일 생성
files_to_zip = []

# 인스타그램 게시물 파일
insta_file = f"{store_info['name']}_인스타그램_{timestamp}.txt"
with open(insta_file, 'w', encoding='utf-8') as f:
    f.write(f"{store_info['name']} 인스타그램 게시물\n{'='*50}\n\n")
    for post in instagram_posts:
        f.write(f"[게시물 {post['number']}\n{post['content']}\n\n")
files_to_zip.append(insta_file)

# 블로그 리뷰 파일
blog_file = f"{store_info['name']}_블로그리뷰_{timestamp}.txt"
with open(blog_file, 'w', encoding='utf-8') as f:
    f.write(f"{store_info['name']} 네이버 블로그 리뷰\n{'='*50}\n\n")
    for i, review in enumerate(blog_reviews, 1):
        f.write(f"[리뷰 {i}: {review['title']}\n{review['content']}\n\n")
files_to_zip.append(blog_file)

# 카카오채널 공지 파일
kakao_file = f"{store_info['name']}_카카오채널_{timestamp}.txt"
with open(kakao_file, 'w', encoding='utf-8') as f:
    f.write(f"{store_info['name']} 카카오채널 공지\n{'='*50}\n\n")
    f.write(f"제목: {kakao_notice['title']}\n\n")
    f.write(f"내용:\n{kakao_notice['content']}\n")
files_to_zip.append(kakao_file)

# 요약 정보 파일
summary_file = f"{store_info['name']}_콘텐츠요약_{timestamp}.txt"
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write(f"{store_info['name']} SNS 콘텐츠 생성 요약\n{'='*50}\n\n")
    f.write(f"생성일시: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(f"가게 정보:\n")
    for key, value in store_info.items():
        f.write(f"  {key}: {value}\n")
    f.write(f"\n추천 메뉴: {', '.join(recommended_menus)}\n\n")
    f.write(f"생성된 콘텐츠:\n")
    f.write(f"  • 인스타그램 게시물: {len(instagram_posts)}개\n")
    f.write(f"  • 블로그 리뷰: {len(blog_reviews)}개\n")
    f.write(f"  • 카카오채널 공지: 1개\n")
    f.write(f"  • 총 파일: {len(files_to_zip)}개\n")
files_to_zip.append(summary_file)

# ZIP 파일 생성
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in files_to_zip:
        zipf.write(file)
        os.remove(file)  # 임시 파일 삭제

print(f"✅ ZIP 파일 생성 완료: {zip_filename}")
print(f"📊 생성된 콘텐츠 요약:")
print(f"  • 인스타그램 게시물: {len(instagram_posts)}개")
print(f"  • 블로그 리뷰: {len(blog_reviews)}개")
print(f"  • 카카오채널 공지: 1개")
print(f"  • 총 {len(files_to_zip)}개 파일 압축 완료")

In [ ]:
# 다운로드 기능 (Colab 환경에서만 작동)
try:
    from google.colab import files
    print("\n📥 ZIP 파일 다운로드")
    files.download(zip_filename)
except ImportError:
    print(f"\n💾 ZIP 파일이 로컬에 저장되었습니다:")
    print(f"- 파일명: {zip_filename}")
    print(f"- 크기: {os.path.getsize(zip_filename):,} bytes")

# 최종 확인
print(f"\n🎉 {store_info['name']} SNS 콘텐츠 자동화 완료!")
print(f"✅ Gemini API {'사용' if gemini_ready else '샘플 데이터'} 모드로 생성 완료")